In [40]:
import pandas as pd
import numpy as np
import math
import numpy as np
from scipy.stats import shapiro 
from scipy.stats import lognorm


In [41]:
data_AU = pd.read_csv('AU_features.csv')


In [42]:
del data_AU['Unnamed: 0']
del data_AU['index']
del data_AU['record']


In [43]:
del data_AU['patient']
del data_AU['timestamp_start']
del data_AU['timestamp_end']

In [44]:
data_AU = data_AU.fillna(data_AU.mean())

In [45]:
data_AU_HC = data_AU[data_AU['diagnosis']==0]

In [46]:
data_AU_dep = data_AU[data_AU['diagnosis']==1]

In [47]:
del data_AU_HC['diagnosis']
del data_AU_dep['diagnosis']

In [48]:
features = list(data_AU_dep.columns)

# Check normality of the data

In [49]:
HC_values = []
dep_values = []

In [50]:
for feature in features:
    HC_values.append(shapiro(data_AU_HC[feature])[1])
    dep_values.append(shapiro(data_AU_dep[feature])[1])

C:\Users\skiju\anaconda3\Lib\site-packages\scipy\stats\_morestats.py:1879: UserWarning: Input data for shapiro has range zero. The results may not be accurate.
  warnings.warn("Input data for shapiro has range zero. The results "


In [12]:
normality_distribution = pd.DataFrame(columns=['features', 'HC_p', 'dep_p'])

In [13]:
normality_distribution['features'] = features

In [14]:
normality_distribution['HC_p'] =HC_values

In [15]:
normality_distribution['dep_p'] = dep_values

In [16]:
normality_distribution[normality_distribution['HC_p']<=0.05]

,features,HC_p,dep_p
0,AU01_mid_Absolute energy,0.001671,0.000070
1,AU01_mid_Area under the curve,0.002916,0.000141
2,AU01_mid_Autocorrelation,0.000005,0.000722
3,AU01_mid_Average power,0.041854,0.496138
4,AU01_mid_Centroid,0.000169,0.004688
...,...,...,...
7095,AU06_eve_rsd,0.000155,0.959659
7096,AU07_eve_rsd,0.000046,0.459381
7097,AU10_eve_rsd,0.009519,0.785761
7098,AU12_eve_rsd,0.000057,0.006704


In [17]:
normality_distribution[normality_distribution['dep_p']<=0.05]

,features,HC_p,dep_p
0,AU01_mid_Absolute energy,0.001671,0.000070
1,AU01_mid_Area under the curve,0.002916,0.000141
2,AU01_mid_Autocorrelation,0.000005,0.000722
4,AU01_mid_Centroid,0.000169,0.004688
5,AU01_mid_ECDF Percentile Count_0,0.000194,0.012516
...,...,...,...
7086,AU12_eve_app_ent,0.013827,0.010803
7088,AU15_eve_app_ent,0.123660,0.004266
7090,AU23_eve_app_ent,0.489309,0.014334
7098,AU12_eve_rsd,0.000057,0.006704


# Since the majority of the features are not normally distributed, Pearson correlation will be calculated

In [51]:
import scipy.stats as stats


In [53]:
data_AU_HC
data_AU_dep

,AU01_mid_Absolute energy,AU01_mid_Area under the curve,AU01_mid_Autocorrelation,AU01_mid_Average power,AU01_mid_Centroid,AU01_mid_ECDF Percentile Count_0,AU01_mid_ECDF Percentile Count_1,AU01_mid_ECDF Percentile_0,AU01_mid_ECDF Percentile_1,AU01_mid_ECDF_0,...,AU04_eve_rsd,AU06_eve_rsd,AU07_eve_rsd,AU10_eve_rsd,AU12_eve_rsd,AU14_eve_rsd,AU15_eve_rsd,AU17_eve_rsd,AU23_eve_rsd,AU24_eve_rsd
2,11.415247,0.220840,2.000000,2.875377,1.133316,79.000000,318.000000,0.000820,0.050790,0.002513,...,265.374413,121.008673,67.176427,179.301774,89.136093,96.213302,138.309344,69.726973,135.622399,202.271039
4,21.142705,0.263578,1.000000,35.237842,0.239713,12.000000,48.000000,0.016091,0.882381,0.016393,...,116.988466,134.024675,46.498004,202.483565,118.604478,92.273881,106.845779,70.164313,109.443031,191.562723
5,20.994168,0.265975,4.000000,29.991668,0.394627,14.000000,56.000000,0.011960,0.858960,0.014085,...,291.131247,162.944912,71.969314,400.045107,112.831481,55.994326,136.362200,128.327725,247.829012,716.936198
10,9.122883,0.145782,2.000000,6.291643,1.045863,29.000000,116.000000,0.000145,0.086164,0.006849,...,176.010829,205.710660,80.940490,334.540653,255.280704,232.573141,224.974399,111.707370,315.072176,756.285165
13,103.357981,1.222127,17.000000,51.421881,0.816389,40.000000,161.000000,0.054901,0.933992,0.004950,...,233.828141,133.229961,74.018833,382.003612,112.218930,111.523354,170.431072,96.982193,148.565087,201.637073
14,143.395653,2.238986,3.000000,12.121357,6.133693,236.000000,947.000000,0.001132,0.413815,0.000845,...,199.171956,147.300449,70.534243,299.204224,151.707252,110.115983,127.825284,53.339761,159.486032,570.492425
15,165.125454,2.305494,3.000000,16.815219,3.968694,196.000000,786.000000,0.001272,0.566184,0.001017,...,251.089074,151.898807,83.133014,289.405013,155.340625,99.134680,124.391735,58.558246,137.297885,433.001841
16,291.831223,3.855189,18.526316,20.541164,6.557825,268.473684,1075.184211,0.009091,0.552215,0.002808,...,189.499509,124.558952,60.143923,236.841595,146.072237,118.307017,156.485998,68.561154,157.643202,515.987555
17,291.831223,3.855189,18.526316,20.541164,6.557825,268.473684,1075.184211,0.009091,0.552215,0.002808,...,189.499509,124.558952,60.143923,236.841595,146.072237,118.307017,156.485998,68.561154,157.643202,515.987555
21,42.924443,0.610312,4.000000,19.780849,0.913781,43.000000,174.000000,0.002914,0.634446,0.004587,...,144.766483,62.805806,39.511453,88.369386,84.034607,90.932349,125.965679,54.331211,102.688696,681.279367


In [54]:
data_AU_dep[features[0]]

2       11.415247
4       21.142705
5       20.994168
10       9.122883
13     103.357981
14     143.395653
15     165.125454
16     291.831223
17     291.831223
21      42.924443
24     138.596402
25     115.728873
31    1416.620799
32    1160.854156
Name: AU01_mid_Absolute energy, dtype: float64

In [55]:
data_AU_test = data_AU

In [56]:
y = data_AU['diagnosis']

In [57]:
del data_AU_test['diagnosis']

In [62]:
r_value = []
p_value = []
mean_value_HC = []
std_value_HC = []
mean_value_dep = []
std_value_dep = []

In [63]:
features = list(data_AU_test.columns)

In [64]:
features

['AU01_mid_Absolute energy',
 'AU01_mid_Area under the curve',
 'AU01_mid_Autocorrelation',
 'AU01_mid_Average power',
 'AU01_mid_Centroid',
 'AU01_mid_ECDF Percentile Count_0',
 'AU01_mid_ECDF Percentile Count_1',
 'AU01_mid_ECDF Percentile_0',
 'AU01_mid_ECDF Percentile_1',
 'AU01_mid_ECDF_0',
 'AU01_mid_ECDF_1',
 'AU01_mid_ECDF_2',
 'AU01_mid_ECDF_3',
 'AU01_mid_ECDF_4',
 'AU01_mid_ECDF_5',
 'AU01_mid_ECDF_6',
 'AU01_mid_ECDF_7',
 'AU01_mid_ECDF_8',
 'AU01_mid_ECDF_9',
 'AU01_mid_Entropy',
 'AU01_mid_Fundamental frequency',
 'AU01_mid_Histogram mode',
 'AU01_mid_Human range energy',
 'AU01_mid_Interquartile range',
 'AU01_mid_Kurtosis',
 'AU01_mid_LPCC_0',
 'AU01_mid_LPCC_1',
 'AU01_mid_LPCC_10',
 'AU01_mid_LPCC_11',
 'AU01_mid_LPCC_2',
 'AU01_mid_LPCC_3',
 'AU01_mid_LPCC_4',
 'AU01_mid_LPCC_5',
 'AU01_mid_LPCC_6',
 'AU01_mid_LPCC_7',
 'AU01_mid_LPCC_8',
 'AU01_mid_LPCC_9',
 'AU01_mid_MFCC_0',
 'AU01_mid_MFCC_1',
 'AU01_mid_MFCC_10',
 'AU01_mid_MFCC_11',
 'AU01_mid_MFCC_2',
 'AU01_m

In [65]:
for feature in features:
    a = stats.pearsonr(data_AU_test[feature], y)[0]
    b = stats.pearsonr(data_AU_test[feature], y)[1]
    r_value.append(abs(a))
    p_value.append(b)
    mean_value_HC.append(np.mean(data_AU_HC[feature]))
    std_value_HC.append(np.std(data_AU_HC[feature]))
    mean_value_dep.append(np.mean(data_AU_dep[feature]))
    std_value_dep.append(np.std(data_AU_dep[feature]))

C:\Users\skiju\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [66]:
statistical_AU = pd.DataFrame(columns=['name_column', '|r-value|', 'p-value', 'HC_mean', 'HC_std', 'dep_mean', 'dep_std'])

In [67]:
statistical_AU['name_column'] = features
statistical_AU['|r-value|'] = r_value
statistical_AU['p-value'] = p_value
statistical_AU['HC_mean'] = mean_value_HC
statistical_AU['HC_std'] = std_value_HC
statistical_AU['dep_mean'] = mean_value_dep
statistical_AU['dep_std'] = std_value_dep

In [71]:
statistical_AU.to_csv('.\dataset\statistical\stat_AU.csv')

# Statistical analysis Euler Angles

In [73]:
data_eul = pd.read_csv('euler_features.csv')


In [74]:
data_eul

,Unnamed: 0,level_0,index,patient,record,diagnosis,timestamp_start,timestamp_end,X_mid_Absolute energy,X_mid_Area under the curve,...,Z_eve_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,Z_eve_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,Z_eve_SP_Summaries_welch_rect_centroid,Z_eve_FC_LocalSimple_mean3_stderr,X_eve_app_ent,Y_eve_app_ent,Z_eve_app_ent,X_eve_rsd,Y_eve_rsd,Z_eve_rsd
0,0,0,0,P08,0,0,1.658354e+09,1.659996e+09,2.385522e+04,17.340800,...,0.183673,0.877551,0.038350,0.556470,1.240145,1.414059,1.101619,-137.406912,247.810229,-84.170855
1,1,1,0,P08,1,0,1.659996e+09,1.661206e+09,2.328345e+05,239.885354,...,0.437500,0.875000,0.104311,0.664134,1.110772,1.153594,1.206691,-246.703278,-917.852063,-147.429543
2,2,2,0,P10,2,1,1.658354e+09,1.659996e+09,4.725678e+04,54.145724,...,0.125000,0.854167,0.193282,0.768328,0.829204,1.076619,1.013430,114.967050,-107.494007,-224.427014
3,3,3,0,P10,3,0,1.659996e+09,1.662070e+09,1.807770e+05,129.773994,...,0.224490,0.122449,0.138058,0.682936,0.659745,0.789076,0.931019,65.528455,-348.921557,-328.457878
4,4,4,0,P12,4,1,1.658441e+09,1.659996e+09,4.557987e+04,24.120677,...,0.860465,0.139535,0.613592,0.924430,0.938069,0.806088,0.934574,72.352631,177.627123,159.713801
5,5,5,0,P12,5,1,1.659996e+09,1.661206e+09,2.937881e+04,14.343478,...,0.619048,0.738095,0.269981,0.758146,0.529294,0.502848,0.595768,44.882796,-1732.345347,174.352004
6,6,6,0,P13,6,0,1.658700e+09,1.659996e+09,1.425495e+05,166.412023,...,0.755102,0.510204,0.036816,0.527443,0.646985,0.687472,0.806980,172.224880,46.271562,73.328927
7,7,7,0,P13,7,0,1.659996e+09,1.661206e+09,1.526638e+04,15.338700,...,0.780488,0.731707,0.220893,0.714463,0.864127,0.889067,0.751850,-108.350642,22.814158,63.237315
8,8,8,0,P14,8,0,1.658700e+09,1.659910e+09,2.917384e+04,22.015482,...,0.448980,0.530612,0.085903,0.636002,1.382823,1.206934,1.187669,318.194246,-125.681839,133.769104
9,9,9,0,P15,9,0,1.658786e+09,1.660082e+09,NaN,NaN,...,0.122449,0.448980,0.171806,0.654882,0.913396,0.750667,0.753536,42.929179,1791.647415,1154.407059


In [75]:
del data_eul['Unnamed: 0']
del data_eul['index']
del data_eul['record']
del data_eul['level_0']

In [76]:
data_eul

,patient,diagnosis,timestamp_start,timestamp_end,X_mid_Absolute energy,X_mid_Area under the curve,X_mid_Autocorrelation,X_mid_Average power,X_mid_Centroid,X_mid_ECDF Percentile Count_0,...,Z_eve_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,Z_eve_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,Z_eve_SP_Summaries_welch_rect_centroid,Z_eve_FC_LocalSimple_mean3_stderr,X_eve_app_ent,Y_eve_app_ent,Z_eve_app_ent,X_eve_rsd,Y_eve_rsd,Z_eve_rsd
0,P08,0,1.658354e+09,1.659996e+09,2.385522e+04,17.340800,16.0,11580.203963,0.936651,41.0,...,0.183673,0.877551,0.038350,0.556470,1.240145,1.414059,1.101619,-137.406912,247.810229,-84.170855
1,P08,0,1.659996e+09,1.661206e+09,2.328345e+05,239.885354,42.0,6007.081693,21.592224,775.0,...,0.437500,0.875000,0.104311,0.664134,1.110772,1.153594,1.206691,-246.703278,-917.852063,-147.429543
2,P10,1,1.658354e+09,1.659996e+09,4.725678e+04,54.145724,63.0,4535.199168,6.523586,208.0,...,0.125000,0.854167,0.193282,0.768328,0.829204,1.076619,1.013430,114.967050,-107.494007,-224.427014
3,P10,0,1.659996e+09,1.662070e+09,1.807770e+05,129.773994,65.0,14002.863964,7.335980,258.0,...,0.224490,0.122449,0.138058,0.682936,0.659745,0.789076,0.931019,65.528455,-348.921557,-328.457878
4,P12,1,1.658441e+09,1.659996e+09,4.557987e+04,24.120677,8.0,30386.579672,0.962864,30.0,...,0.860465,0.139535,0.613592,0.924430,0.938069,0.806088,0.934574,72.352631,177.627123,159.713801
5,P12,1,1.659996e+09,1.661206e+09,2.937881e+04,14.343478,2.0,34974.776061,0.392917,17.0,...,0.619048,0.738095,0.269981,0.758146,0.529294,0.502848,0.595768,44.882796,-1732.345347,174.352004
6,P13,0,1.658700e+09,1.659996e+09,1.425495e+05,166.412023,56.0,4206.240544,18.322024,678.0,...,0.755102,0.510204,0.036816,0.527443,0.646985,0.687472,0.806980,172.224880,46.271562,73.328927
7,P13,0,1.659996e+09,1.661206e+09,1.526638e+04,15.338700,11.0,6131.074397,0.533629,50.0,...,0.780488,0.731707,0.220893,0.714463,0.864127,0.889067,0.751850,-108.350642,22.814158,63.237315
8,P14,0,1.658700e+09,1.659910e+09,2.917384e+04,22.015482,8.0,15354.652608,1.082417,38.0,...,0.448980,0.530612,0.085903,0.636002,1.382823,1.206934,1.187669,318.194246,-125.681839,133.769104
9,P15,0,1.658786e+09,1.660082e+09,NaN,NaN,NaN,NaN,NaN,NaN,...,0.122449,0.448980,0.171806,0.654882,0.913396,0.750667,0.753536,42.929179,1791.647415,1154.407059


In [77]:
del data_eul['patient']
del data_eul['timestamp_start']
del data_eul['timestamp_end']

In [79]:
data_eul = data_eul.fillna(data_eul.mean())

In [80]:
data_eul_HC = data_eul[data_eul['diagnosis']==0]
data_eul_dep = data_eul[data_eul['diagnosis']==1]

In [81]:
del data_eul_HC['diagnosis']
del data_eul_dep['diagnosis']

In [82]:
data_eul_test = data_eul
y = data_eul['diagnosis']
del data_eul_test['diagnosis']

In [83]:
r_value = []
p_value = []
mean_value_HC = []
std_value_HC = []
mean_value_dep = []
std_value_dep = []

In [84]:
features = list(data_eul_test.columns)

In [86]:
for feature in features:
    a = stats.pearsonr(data_eul_test[feature], y)[0]
    b = stats.pearsonr(data_eul_test[feature], y)[1]
    r_value.append(abs(a))
    p_value.append(b)
    mean_value_HC.append(np.mean(data_eul_HC[feature]))
    std_value_HC.append(np.std(data_eul_HC[feature]))
    mean_value_dep.append(np.mean(data_eul_dep[feature]))
    std_value_dep.append(np.std(data_eul_dep[feature]))

In [87]:
statistical_eul = pd.DataFrame(columns=['name_column', '|r-value|', 'p-value', 'HC_mean', 'HC_std', 'dep_mean', 'dep_std'])

In [88]:
statistical_eul['name_column'] = features
statistical_eul['|r-value|'] = r_value
statistical_eul['p-value'] = p_value
statistical_eul['HC_mean'] = mean_value_HC
statistical_eul['HC_std'] = std_value_HC
statistical_eul['dep_mean'] = mean_value_dep
statistical_eul['dep_std'] = std_value_dep

In [89]:
statistical_eul.to_csv('.\dataset\statistical\stat_eul.csv')

# Eye-opening and smiling probabilities

In [93]:
data_prob = pd.read_csv('prob_eye_features.csv')


In [95]:
del data_prob['Unnamed: 0']
del data_prob['index']
del data_prob['record']


In [96]:
del data_prob['patient']
del data_prob['timestamp_start']
del data_prob['timestamp_end']

In [97]:
data_prob = data_prob.fillna(data_prob.mean())

In [98]:
data_prob_HC = data_prob[data_prob['diagnosis']==0]
data_prob_dep = data_prob[data_prob['diagnosis']==1]

In [99]:
del data_prob_HC['diagnosis']
del data_prob_dep['diagnosis']

In [100]:
data_prob_test = data_prob
y = data_prob['diagnosis']
del data_prob_test['diagnosis']

In [101]:
r_value = []
p_value = []
mean_value_HC = []
std_value_HC = []
mean_value_dep = []
std_value_dep = []

In [102]:
features = list(data_prob_test.columns)

In [103]:
for feature in features:
    a = stats.pearsonr(data_prob_test[feature], y)[0]
    b = stats.pearsonr(data_prob_test[feature], y)[1]
    r_value.append(abs(a))
    p_value.append(b)
    mean_value_HC.append(np.mean(data_prob_HC[feature]))
    std_value_HC.append(np.std(data_prob_HC[feature]))
    mean_value_dep.append(np.mean(data_prob_dep[feature]))
    std_value_dep.append(np.std(data_prob_dep[feature]))

C:\Users\skiju\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [104]:
statistical_prob = pd.DataFrame(columns=['name_column', '|r-value|', 'p-value', 'HC_mean', 'HC_std', 'dep_mean', 'dep_std'])

In [105]:
statistical_prob['name_column'] = features
statistical_prob['|r-value|'] = r_value
statistical_prob['p-value'] = p_value
statistical_prob['HC_mean'] = mean_value_HC
statistical_prob['HC_std'] = std_value_HC
statistical_prob['dep_mean'] = mean_value_dep
statistical_prob['dep_std'] = std_value_dep

In [106]:
statistical_prob.to_csv('.\dataset\statistical\stat_prob.csv')

# Facial features

In [107]:
important_features = list(pd.read_csv('.\dataset\important_features.csv', header=None)[0])

In [108]:
import polars as pl
import csv


In [109]:
file_path = './dataset/ff/data_all_ff.csv'

In [111]:
#Use scan_csv and select to read specific columns
df = pl.scan_csv(file_path).select(important_features).collect()


In [112]:
columns_to_fill = list(df.columns[:-3])


In [113]:
df_filled = df.with_columns(
    [pl.when(pl.col(col).is_infinite()).then(None).otherwise(pl.col(col)).alias(col) for col in columns_to_fill]
    
)

In [114]:


# Fill NaN values with the mean for the specified columns
df_filled = df_filled.with_columns(
    [pl.col(col).fill_nan(pl.col(col).mean()) for col in columns_to_fill]
)

print(df_filled)


shape: (43, 14_663)
┌────────────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬────────┬─────────┐
│ 0_13distan ┆ 0_104angle ┆ 0_104angle ┆ 0_101angle ┆ … ┆ 93_103ang ┆ diagnosis ┆ record ┆ patient │
│ ce_mid_Hum ┆ _mid_MFCC_ ┆ _mid_Max   ┆ _mid_Spect ┆   ┆ le_eve_rs ┆ ---       ┆ ---    ┆ ---     │
│ an range   ┆ 5          ┆ ---        ┆ ral        ┆   ┆ d         ┆ i64       ┆ i64    ┆ str     │
│ e…         ┆ ---        ┆ f64        ┆ kurtos…    ┆   ┆ ---       ┆           ┆        ┆         │
│ ---        ┆ f64        ┆            ┆ ---        ┆   ┆ f64       ┆           ┆        ┆         │
│ f64        ┆            ┆            ┆ f64        ┆   ┆           ┆           ┆        ┆         │
╞════════════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪════════╪═════════╡
│ 0.00325    ┆ -1.840351  ┆ 1.568902   ┆ 2.212502   ┆ … ┆ -107.0520 ┆ 0         ┆ 0      ┆ P08     │
│            ┆            ┆            ┆            ┆   ┆ 95        ┆  

In [115]:
df_filled2 = df_filled.with_columns(
    [pl.col(col).fill_null(pl.col(col).mean()) for col in columns_to_fill]
)

In [116]:
X = df_filled2[df_filled2.columns[:-3]]

In [117]:
X = X.to_pandas()

In [118]:
group = df_filled2['patient'].to_pandas()

In [119]:
groups = group

In [120]:
y = df_filled2['diagnosis'].to_pandas()

In [121]:
records  = df_filled2['record'].to_pandas()

In [124]:
data_ff = pd.concat([y, X], axis=1)


,diagnosis,0_13distance_mid_Human range energy,0_104angle_mid_MFCC_5,0_104angle_mid_Max,0_101angle_mid_Spectral kurtosis,0_16distance_mid_ECDF_0,0_127angle_mid_Spectral decrease,0_107angle_mid_Max power spectrum,0_122distance_mid_Median diff,0_123angle_mid_MFCC_1,...,55_105angle_eve_rsd,44_54angle_eve_rsd,70_82distance_eve_rsd,43_51angle_eve_rsd,109_117angle_eve_rsd,59_69angle_eve_rsd,110_111angle_eve_rsd,64_124angle_eve_rsd,106_109angle_eve_rsd,93_103angle_eve_rsd
0,0,0.003250,-1.840351,1.568902,2.212502,0.004831,-1.079578,0.545417,-0.010843,-68.433294,...,-51.343138,67.763495,1.382429,52.873894,46.697858,202.789080,1383.660106,-58.588626,178.580409,-107.052095
1,0,0.000955,10.081815,1.570796,1.898060,0.000258,-0.272252,5.245545,0.001702,-24.671142,...,-106.718558,114.804993,1.675652,72.912386,55.323645,197.969723,-495.127710,-110.520797,144.754258,-325.967288
2,1,0.000110,69.328889,1.570796,2.235668,0.001130,-0.785619,0.147481,-0.000486,-22.544975,...,-145.589786,645.392595,1.021050,44.300970,26.021285,164.124128,11587.365133,-101.816203,26.929642,62.154620
3,0,0.000263,78.030851,1.570796,2.323626,0.000897,-0.259407,0.051975,0.001393,-20.399725,...,-284.593329,912.118427,1.199724,46.470249,30.388807,-525.380688,196.086475,-119.914317,36.841969,137.130355
4,1,0.000042,-27.148089,1.570796,1.861678,0.006623,-0.767740,0.023533,0.002480,-8.105020,...,310.271316,-282.929401,1.952581,287.355117,61.243625,1082.071061,-857.901089,434.934293,67.797981,295.450994
5,1,0.000469,14.970590,1.568357,2.026775,0.011765,-1.149265,0.030957,-0.000262,-86.000882,...,147.260741,-155.883294,1.473812,427.659922,64.214727,514.732773,-395.997262,161.851396,70.235880,117.903253
6,0,0.000350,40.421336,1.568454,2.085611,0.000295,-1.375563,0.713760,0.001193,-21.588570,...,55.500204,-214.075757,1.297797,107.747378,36.554223,78.725785,-103.424195,68.516986,29.666199,-173.979804
7,0,0.000108,28.664611,1.570796,3.358030,0.004000,-0.592514,0.703830,-0.001916,-31.573657,...,88.526928,-2562.948886,0.977047,53.012193,22.533279,4.457467,-32.598258,42.621967,15.453300,-139.264115
8,0,0.000198,12.507607,1.553527,1.742731,0.005236,-0.190092,0.008482,0.002279,-17.564342,...,128.583960,-90.187301,1.850571,-511.598888,88.994644,69.059974,-81.258403,272.404397,92.216660,65.162159
9,0,0.000918,32.330542,1.570056,2.210358,0.001453,-0.331715,1.032832,0.000133,-18.094546,...,-388.031551,1536.097858,1.193838,62.420082,44.912848,-108.463034,719.874536,-221.930210,33.245194,278.504680


In [125]:
data_ff_HC = data_ff[data_ff['diagnosis']==0]
data_ff_dep = data_ff[data_ff['diagnosis']==1]

In [126]:
del data_ff_HC['diagnosis']
del data_ff_dep['diagnosis']

In [127]:
data_ff_test = data_ff
y = data_ff['diagnosis']
del data_ff_test['diagnosis']

In [128]:
r_value = []
p_value = []
mean_value_HC = []
std_value_HC = []
mean_value_dep = []
std_value_dep = []

In [129]:
features = list(data_ff_test.columns)

In [130]:
for feature in features:
    a = stats.pearsonr(data_ff_test[feature], y)[0]
    b = stats.pearsonr(data_ff_test[feature], y)[1]
    r_value.append(abs(a))
    p_value.append(b)
    mean_value_HC.append(np.mean(data_ff_HC[feature]))
    std_value_HC.append(np.std(data_ff_HC[feature]))
    mean_value_dep.append(np.mean(data_ff_dep[feature]))
    std_value_dep.append(np.std(data_ff_dep[feature]))

C:\Users\skiju\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [131]:
statistical_ff = pd.DataFrame(columns=['name_column', '|r-value|', 'p-value', 'HC_mean', 'HC_std', 'dep_mean', 'dep_std'])

In [132]:
statistical_ff['name_column'] = features
statistical_ff['|r-value|'] = r_value
statistical_ff['p-value'] = p_value
statistical_ff['HC_mean'] = mean_value_HC
statistical_ff['HC_std'] = std_value_HC
statistical_ff['dep_mean'] = mean_value_dep
statistical_ff['dep_std'] = std_value_dep

In [133]:
statistical_ff.to_csv('.\dataset\statistical\stat_ff.csv')

# Concat all modalities

In [137]:
statistical_analysis = pd.concat([statistical_AU, statistical_eul, statistical_prob, statistical_ff], axis=0)

In [138]:
statistical_analysis

,name_column,|r-value|,p-value,HC_mean,HC_std,dep_mean,dep_std
0,AU01_mid_Absolute energy,0.022678,0.885228,297.096599,280.719671,280.924372,423.752479
1,AU01_mid_Area under the curve,0.035639,0.820511,3.956729,3.578449,3.644858,5.005329
2,AU01_mid_Autocorrelation,0.283366,0.065571,23.675136,30.034361,7.860902,7.956739
3,AU01_mid_Average power,0.077808,0.619938,19.868842,11.545548,21.933829,14.001060
4,AU01_mid_Centroid,0.122723,0.433037,7.091934,6.521866,5.451457,5.530448
...,...,...,...,...,...,...,...
14655,59_69angle_eve_rsd,0.374128,0.013454,116.031809,364.004756,568.743867,758.160854
14656,110_111angle_eve_rsd,0.040315,0.797425,201.449759,941.660407,370.365802,3159.637408
14657,64_124angle_eve_rsd,0.082760,0.597769,702.099057,3651.736529,168.873980,426.637712
14658,106_109angle_eve_rsd,0.039867,0.799634,72.220791,46.255291,68.874693,17.650703


In [143]:
statistical_analysis_reorderd = statistical_analysis.sort_values(by=['|r-value|'], ascending=False)

In [144]:
statistical_analysis_reorderd = statistical_analysis_reorderd.reset_index()

In [146]:
del statistical_analysis_reorderd['index']

In [153]:
statistical_analysis_reorderd['Depressive Episode Mean (SD)'] = 0
statistical_analysis_reorderd['Non-Depressive State Mean (SD)'] = 0


In [172]:
sub_set_statistical = statistical_analysis_reorderd.loc[0:500]

In [173]:
for sample in range(0, len(sub_set_statistical)):
    sub_set_statistical['Depressive Episode Mean (SD)'].loc[sample] = str(round(sub_set_statistical['dep_mean'].loc[sample], 2)) + ' (' + str(round(sub_set_statistical['dep_std'].loc[sample], 2)) + ')'
    sub_set_statistical['Non-Depressive State Mean (SD)'].loc[sample] = str(round(sub_set_statistical['HC_mean'].loc[sample], 2)) + ' (' + str(round(sub_set_statistical['HC_std'].loc[sample], 2)) + ')'

C:\Users\skiju\AppData\Local\Temp\ipykernel_8068\734779192.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_set_statistical['Depressive Episode Mean (SD)'].loc[sample] = str(round(sub_set_statistical['dep_mean'].loc[sample], 2)) + ' (' + str(round(sub_set_statistical['dep_std'].loc[sample], 2)) + ')'
C:\Users\skiju\AppData\Local\Temp\ipykernel_8068\734779192.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_set_statistical['Non-Depressive State Mean (SD)'].loc[sample] = str(round(sub_set_statistical['HC_mean'].loc[sample], 2)) + ' (' + str(round(sub_set_statistical['HC_std'].loc[sample], 2)) + ')'
C:\Users

In [175]:
sub_set_statistical.to_csv('.\dataset\statistical\statistical_features500.csv')